# 🚀 MorphoSpeech-LLM: State-of-the-Art Turkish Automatic Speech Recognition (ASR)

**MorphoSpeech-LLM** is a novel morphology-aware Audio-LLM framework designed to surpass current SOTA benchmarks on Turkish ASR datasets (**Mozilla Common Voice** and **Google FLEURS**).

### 🌟 Key Innovations:
1. **Turkish Morpho-Tactic Tokenizer (`MorphoTokenizer`)**: Morfessor + BPE integration for agglutinative root-suffix decomposition.
2. **Auxiliary CTC Loss Guidance**: Dual-objective optimization (CTC + Cross-Attention) to ground acoustic phonemes.
3. **Turkish Text Normalization (`TurkishTextNormalizer`)**: Standardized I/İ casing, number-to-words expansion, and punctuation normalization.
4. **PEFT/LoRA Fine-Tuning on NVIDIA A100 GPU**: Memory-efficient, high-speed training.

## 🛠️ Step 1: Environment Setup & Hardware Check

In [1]:
# Check GPU Availability (Target: NVIDIA A100 GPU)
!nvidia-smi

# Install Required Packages
!pip install -q --upgrade transformers datasets[audio] peft bitsandbytes evaluate jiwer morfessor subword-nmt librosa soundfile accelerate huggingface_hub

# Load Hugging Face Access Token from Colab Secrets (🔑 Gizli Anahtarlar tab)
import os
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
    if token:
        os.environ['HF_TOKEN'] = token
        os.environ['HUGGING_FACE_HUB_TOKEN'] = token
        from huggingface_hub import login
        login(token=token)
        print('🔑 HF_TOKEN Colab Secrets\'tan başarıyla yüklendi ve HF Hub\'a giriş yapıldı!')
    else:
        print('ℹ️ HF_TOKEN gizli anahtarı boş. Açık verisetleri (Google FLEURS) kullanılacak.')
except Exception as e:
    if os.environ.get('HF_TOKEN'):
        print('🔑 HF_TOKEN ortam değişkeninden yüklendi!')
    else:
        print('ℹ️ HF_TOKEN olmadan anonim olarak devam ediliyor (Google FLEURS kullanılacak).')

Sun Oct  4 20:29:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


🔑 HF_TOKEN Colab Secrets'tan başarıyla yüklendi ve HF Hub'a giriş yapıldı!


## 📂 Step 2: Project Setup & Repository Cloning

In [2]:
import os
import sys

# Clone repository if running directly in Google Colab
if not os.path.exists('src'):
    !git clone https://github.com/gokhanersoy/turkish-asr.git
    %cd turkish-asr

sys.path.append(os.getcwd())
print("Current Working Directory:", os.getcwd())

Cloning into 'turkish-asr'...
remote: Enumerating objects: 231, done.
remote: Counting objects: 100% (231/231), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 231 (delta 96), reused 205 (delta 70), pack-reused 0 (from 0)
Receiving objects: 100% (231/231), 146.38 KiB | 13.31 MiB/s, done.
Resolving deltas: 100% (96/96), done.
/content/turkish-asr
Current Working Directory: /content/turkish-asr


## 📊 Step 3: Baseline Evaluation (Whisper Large-v3 Zero-Shot)
Evaluate baseline model performance on **Mozilla Common Voice (Turkish)** or **Google FLEURS (Turkish)** before fine-tuning.

In [3]:
import torch
from src.utils.metrics import compute_wer_cer
from src.data.dataset_loader import TurkishASRDatasetLoader
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "openai/whisper-large-v3"

print(f"Loading Baseline Model: {model_id} on {device}...")
processor = WhisperProcessor.from_pretrained(model_id, language="turkish", task="transcribe")
baseline_model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    low_cpu_mem_usage=True
).to(device)

data_loader = TurkishASRDatasetLoader()
# Will load Common Voice if token is available, or seamlessly fallback to Google FLEURS (ungated)
test_dataset = data_loader.load_common_voice(split="test", max_samples=50)

preds, refs = [], []
for sample in tqdm(test_dataset, desc="Evaluating Baseline Zero-Shot"):
    audio = sample["audio"]
    ref_text = sample.get("sentence") or sample.get("transcription") or sample.get("text") or sample.get("raw_transcription") or ""
    input_features = processor(audio["array"], sampling_rate=audio["sampling_rate"], return_tensors="pt").input_features.to(device).half()
    with torch.no_grad():
        predicted_ids = baseline_model.generate(input_features, forced_decoder_ids=processor.get_decoder_prompt_ids(language="turkish", task="transcribe"))
    transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    preds.append(transcription)
    refs.append(ref_text)

baseline_metrics = compute_wer_cer(preds, refs)
print(f"\nBaseline Zero-Shot Results:")
print(f" - WER: {baseline_metrics['wer']}%")
print(f" - CER: {baseline_metrics['cer']}%")

Loading Baseline Model: openai/whisper-large-v3 on cuda...


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/2526 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/338 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/743 [00:00<?, ? examples/s]

Evaluating Baseline Zero-Shot:   0%|          | 0/50 [00:00<?, ?it/s][transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
Evaluating Baseline Zero-Shot: 100%|██████████| 50/50 [01:24<00:0


Baseline Zero-Shot Results:
 - WER: 7.47%
 - CER: 1.28%


## 🏋️ Step 4: Fine-Tuning MorphoSpeech-LLM on NVIDIA A100 GPU
Train MorphoSpeech-LLM using PEFT/LoRA and Auxiliary CTC loss.

In [4]:
!PYTHONPATH=. python src/train.py \
    --model_name_or_path openai/whisper-large-v3 \
    --dataset_name mozilla-foundation/common_voice_13_0 \
    --dataset_config tr \
    --output_dir ./checkpoints/morpho_speech_llm_a100 \
    --per_device_train_batch_size 16 \
    --per_device_eval_batch_size 16 \
    --learning_rate 2e-4 \
    --num_train_epochs 3 \
    --gradient_accumulation_steps 2 \
    --fp16 \
    --use_lora

Traceback (most recent call last):
  File "/content/turkish-asr/src/train.py", line 8, in <module>
    import evaluate
  File "/usr/local/lib/python3.13/dist-packages/evaluate/__init__.py", line 29, in <module>
    from .evaluation_suite import EvaluationSuite
  File "/usr/local/lib/python3.13/dist-packages/evaluate/evaluation_suite/__init__.py", line 7, in <module>
    from datasets import Dataset, DownloadConfig, DownloadMode, load_dataset
ImportError: cannot import name 'Dataset' from 'datasets' (/content/turkish-asr/src/datasets/__init__.py)


## 🏆 Step 5: Benchmark & SOTA Comparison
Compare MorphoSpeech-LLM results against existing literature SOTA values.

In [5]:
import pandas as pd

sota_results = [
    {"Model": "Whisper Large-v3 (Zero-Shot)", "Dataset": "Common Voice (TR)", "WER (%)": 8.9, "Paper/Source": "OpenAI (2023)"},
    {"Model": "Meta MMS-1B", "Dataset": "FLEURS (TR)", "WER (%)": 7.2, "Paper/Source": "Meta AI (2023)"},
    {"Model": "Whisper-Small LoRA", "Dataset": "Common Voice (TR)", "WER (%)": 16.0, "Paper/Source": "DergiPark (2023)"},
    {"Model": "HuBERT-TR", "Dataset": "Broadcast News", "WER (%)": 4.97, "Paper/Source": "HuggingFace (2023)"},
    {"Model": "MorphoSpeech-LLM (Ours)", "Dataset": "Common Voice / FLEURS", "WER (%)": "< 5.5 (Target)", "Paper/Source": "MorphoSpeech-LLM Framework"}
]

df = pd.DataFrame(sota_results)
print(df.to_markdown(index=False))

| Model                        | Dataset               | WER (%)        | Paper/Source               |
|:-----------------------------|:----------------------|:---------------|:---------------------------|
| Whisper Large-v3 (Zero-Shot) | Common Voice (TR)     | 8.9            | OpenAI (2023)              |
| Meta MMS-1B                  | FLEURS (TR)           | 7.2            | Meta AI (2023)             |
| Whisper-Small LoRA           | Common Voice (TR)     | 16.0           | DergiPark (2023)           |
| HuBERT-TR                    | Broadcast News        | 4.97           | HuggingFace (2023)         |
| MorphoSpeech-LLM (Ours)      | Common Voice / FLEURS | < 5.5 (Target) | MorphoSpeech-LLM Framework |
